# 00 · Tour: the loop, the stack, the scenario

This series walks through the code behind the tutorial, one idea per notebook:

| Notebook | What you'll see |
|---|---|
| **00 · Tour** (this one) | how the repo is configured, the scenario, and the Prompt Registry |
| **01 · Agents and traces** | one question answered by OpenClaw, Hermes Agent, and an offline stand-in, all producing the same trace shape |
| **02 · Signal** | reviewers, code checks, tool-level checks, and LLM judges, as MLflow assessments |
| **03 · Store, update, gate** | dataset → candidate → gate → promote → rollback, for `AGENTS.md` and for skills |
| **04 · The loop runs itself** | Level 5: drift arrives, and the loop notices, fixes, gates, and ships |

Each notebook runs on its own, against a local MLflow store in `notebooks/.mlflow/`.
* **The agent** runs on the **offline** runtime (a rule-based stand-in for the LLM) unless a cell asks for OpenClaw
  or Hermes.
* **Judges and the analyst** run on Claude when `ANTHROPIC_API_KEY` is set:
  * Claude Haiku 5.5 is the support agent;
  * Claude Opus 5.5 judges and analyzes.
* **Fallbacks.** Cells that need a runtime or a model check first and skip themselves with a reason. Notebooks 03
  and 04 pass `llm=False` so their outputs are deterministic.

To browse what the notebooks create, run `make notebooks-ui` and open http://127.0.0.1:5001.

## The loop every level runs

![loop](../docs/architecture/svg/01-loop.svg)

Signal → store → update → gate → promote. Levels 0–5 change only **who writes the candidate** and **who approves
it**.

## Setup

`sia.notebook.setup()` has to run before anything else from `sia` is imported, because `sia.config` reads the
environment once. `fresh=True` wipes the shared notebook store, which makes this the place to reset the series.

In [ ]:
from sia import notebook as nb

nb.setup(backend="offline", fresh=True)

In [ ]:
# Which agent runtimes can run on this machine right now?
nb.available_backends()

## One model naming scheme, four runtimes

Every model is an MLflow URI, `provider:/model`. MLflow judges and optimizers take it as-is. `sia.llm` translates
it for LiteLLM (the `direct` runtime and the analyst), for OpenClaw, and for Hermes. `SIA_PROVIDER` picks a preset
pair: a small agent model and a stronger judge model.

In [ ]:
import pandas as pd
from sia import config, llm

print("agent:", config.AGENT_MODEL, "   judge:", config.JUDGE_MODEL)
pd.DataFrame([
    {"provider": p, "agent": a, "judge": j, "LiteLLM id": llm.litellm_model(a), "OpenClaw id": llm.openclaw_model(a)}
    for p, (a, j) in llm.PRESETS.items()
])

In [ ]:
# Hermes talks to every provider through its OpenAI-compatible endpoint. For Ollama it uses a 64K-context
# alias of the model, because Ollama's /v1 endpoint ignores per-request context size.
llm.hermes_endpoint("ollama:/gemma4:e4b")

## The scenario: Acme Cloud support

Twenty-four customer questions in four splits:
* `traffic` becomes production traces.
* `holdout` is never shown to an optimizer.
* `drift` starts arriving later, about a policy file the instructions don't know about.

The `must_include` / `must_not_include` patterns are the simulated reviewers' private answer key.

In [ ]:
from sia.scenario import load_cases

cases = pd.DataFrame([c.__dict__ for c in load_cases()])
cases.groupby(["split", "topic"]).size().unstack(fill_value=0)

In [ ]:
cases[["id", "split", "topic", "question", "expected_response"]].head(8)

The agent's knowledge base is the source of truth. Its *instructions*, `AGENTS.md`, start out flawed on
purpose: stale facts, a mandatory disclaimer, and "give your best guess".

In [ ]:
from sia.workspace import seed_agents_md

print(seed_agents_md())

## Versioning the thing we'll change: the Prompt Registry

`AGENTS.md` lives in MLflow's Prompt Registry. Production is whatever the `@production` alias points at, so
promoting or rolling back is just moving an alias.

In [ ]:
import mlflow
from sia.agent import load_agents_md

v = nb.ensure_production()            # registers v1 and points @production at it (once)
prod = load_agents_md("production")
print(f"{config.PROMPT_NAME} v{prod.version} is @production")
print(prod.template[:300], "...")

## Next

Open **01 · Agents and traces** to run the agent and look at what a trace contains.